# Session 7: Cell types

A cell-by-gene table is where most single-cell analysis starts, and spatial data adds a position to every row. In
this session you will:

1. prepare the two tables in the workshop data: transcript counts (RNA) and protein intensities;
2. group the cells into **clusters** by their RNA, and give the clusters names using known marker genes;
3. see where each cell type is in the tissue;
4. compare cell types from RNA with the protein measured in the same cells;
5. compare them with the phenotypes from QuPath on Day 1.

We work through all of it first on **10x's cells**, so that everyone sees the same example. In the last exercise you
repeat it on **your own work**: the InstanSeg cells you saved in session 6, and your Day 1 phenotypes.

The tools are [scanpy](https://scanpy.readthedocs.io), the standard Python toolkit for single-cell data, and the
AnnData tables you met in session 6.

Cells marked **Exercise** are for you to complete. The completed notebook is in `solutions/`.

## Setup

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import spatialdata as sd
from sklearn.metrics import adjusted_rand_score

DATA = Path("../data")
sdata = sd.read_zarr(DATA / "xenium_rcc_crop.zarr")
PIXEL_SIZE = sdata.attrs["pixel_size_um"]  # µm per image pixel

rna = sdata["table"].copy()  # transcript counts, read into memory (the tables are small)
protein = sdata["protein"].copy()  # mean protein intensities
print(rna)
print(protein)

## 1. Two tables, two kinds of number

Both tables have one row per 10x cell, in the same order, but they hold different kinds of number:

- `rna` holds **counts**: how many transcripts of each gene were found in the cell. Counts depend on the cell's size,
  how much of it was in the section, and how efficiently its RNA was detected.
- `protein` holds **intensities**: the mean brightness of each antibody channel over the cell's pixels.

So they need different preparation, and we keep them in separate tables. First, how many transcripts does each cell
have?

In [ ]:
counts_per_cell = np.asarray(rna.X.sum(axis=1)).ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].hist(counts_per_cell, bins=np.arange(0, 400, 5))
axes[0].set(xlabel="transcripts per cell", ylabel="cells")
axes[1].hist(protein[:, "CD3E"].X.toarray().ravel(), bins=100, log=True)
axes[1].set(xlabel="mean CD3E intensity per cell", ylabel="cells")
plt.show()

A cell with only a handful of transcripts can't be classified reliably, so we leave out cells with fewer than 20.
Here that removes about one cell in ten. Some of them are real cells, cut at the surface of the section or with
little RNA, so it's worth checking where they are (we will, in section 4).

In [ ]:
MIN_TRANSCRIPTS = 20

sc.pp.filter_cells(rna, min_counts=MIN_TRANSCRIPTS)
protein = protein[rna.obs_names].copy()  # keep the same cells in both tables
print(f"{rna.n_obs:,} of {len(counts_per_cell):,} cells have at least {MIN_TRANSCRIPTS} transcripts")

### Normalisation

- **RNA:** scale each cell's counts to the same total, so that big cells don't look like they express everything more,
  then take the logarithm, `log(1 + x)`, so that a few very abundant genes don't dominate.
- **Protein:** intensities don't need scaling to a total. A common transform for antibody data is `arcsinh(x / 5)`,
  which behaves like a logarithm for bright values but stays smooth near 0, where most cells are. Each marker is then
  scaled to mean 0 and standard deviation 1, so bright and dim antibodies count equally.

We write each as a function so that we can reuse it on your own cells later. The raw values are kept in a **layer**,
in case we need them.

In [ ]:
def normalise_rna(adata):
    """Scale each cell to the same total count, then log(1 + x). The raw counts are kept in layers["counts"]."""
    adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)


def normalise_protein(adata, cofactor=5):
    """arcsinh(x / cofactor), then scale each marker. The raw intensities are kept in layers["intensity"]."""
    values = adata.X.toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X)
    adata.layers["intensity"] = values.copy()
    adata.X = np.arcsinh(values / cofactor)
    sc.pp.scale(adata, max_value=10)


normalise_rna(rna)
normalise_protein(protein)

## 2. Clusters from RNA

Clustering groups cells whose expression profiles are similar. The usual scanpy route has four steps:

1. **PCA** summarises the 405 genes as 30 principal components, which keeps the main patterns and drops much of the
   noise;
2. **neighbours:** each cell is linked to the 15 cells most similar to it, giving a graph;
3. **Leiden** finds groups of cells that are more connected to each other than to the rest. Its `resolution` sets how
   many groups you get;
4. **UMAP** draws the graph in two dimensions, for looking at, not for measuring: distances on a UMAP are distorted.

On a laptop this takes about a minute.

In [ ]:
def cluster(adata, resolution=0.5, use_pca=True):
    """PCA (optional), neighbour graph, UMAP and Leiden clusters (in adata.obs["leiden"])."""
    if use_pca:
        sc.pp.pca(adata, n_comps=30)
        sc.pp.neighbors(adata, n_neighbors=15)
    else:  # few features (e.g. 27 proteins): use them directly
        sc.pp.neighbors(adata, n_neighbors=15, use_rep="X")
    sc.tl.umap(adata)
    sc.tl.leiden(adata, resolution=resolution, flavor="igraph", n_iterations=2, directed=False)


cluster(rna)
print(f"{rna.obs['leiden'].nunique()} clusters")
sc.pl.umap(rna, color="leiden", legend_loc="on data", title="RNA clusters")

**Exercise 1.** Run Leiden again with `resolution=0.2` and `resolution=1.0`, saving the results under new names with
`key_added` (for example `"leiden_0.2"`). How many clusters do you get each time? Plot all three on the UMAP. Which
clusters split or merge?

In [ ]:
for resolution in (0.2, 1.0):
    sc.tl.leiden(rna, resolution=resolution, key_added=f"leiden_{resolution}", flavor="igraph", n_iterations=2,
                 directed=False)
    print(f"resolution {resolution}: {rna.obs[f'leiden_{resolution}'].nunique()} clusters")
sc.pl.umap(rna, color=["leiden_0.2", "leiden", "leiden_1.0"], legend_loc="on data")

**Questions**

- Is there a "right" number of clusters? What would you need to know to choose one?
- At high resolution, are the new small clusters new cell types, or states of the same type?

We carry on with the default, `resolution=0.5` (the `leiden` column).

## 3. Naming the clusters

Clusters are numbered, not named. To name them we use **marker genes**: genes known to be expressed by one cell type
and not others. Each list below has a few markers for one cell type, chosen from the genes in this panel.

This panel has no classic markers of clear-cell renal carcinoma (such as *CA9*), so the tumour list uses genes that
are high in the tumour cells here: *MET*, N-cadherin (*CDH2*), keratin 8 and caveolin 1. That's a common situation
with targeted panels: check what your panel can and can't tell apart before you start.

In [ ]:
MARKERS = {
    "T cell": ["CD3E", "CD3D", "CD2", "TRAC"],
    "B cell": ["MS4A1", "CD79A", "CD19", "BANK1"],
    "Plasma cell": ["MZB1", "PRDM1", "SDC1", "SLAMF7"],
    "Macrophage": ["CD68", "CD163", "MS4A6A", "FCGR3A"],
    "Blood endothelium": ["PECAM1", "VWF", "CD34", "CLEC14A"],
    "Lymphatic endothelium": ["PROX1", "LYVE1", "MMRN1", "TFPI"],
    "Smooth muscle": ["ACTA2", "PDGFRB", "MYH11", "MYLK"],
    "Fibroblast": ["PDGFRA", "C7", "TNC", "FBN1"],
    "Tumour": ["MET", "CDH2", "KRT8", "CAV1"],
}
sc.pl.dotplot(rna, MARKERS, groupby="leiden", standard_scale="var")

In the dot plot, each row is a cluster and each column a gene: the colour is the gene's mean expression in the
cluster, and the size of the dot the fraction of cells that express it. Most clusters light up one block of markers.

Naming by eye works, but doesn't scale and isn't written down. `name_clusters` does the same thing in code: it scores
every cell for each list of markers (`sc.tl.score_genes`), and gives each cluster the name with the highest mean
score.

In [ ]:
def name_clusters(adata, markers=MARKERS, key="leiden"):
    """Name each cluster after the marker list with the highest mean score. Returns the names and the scores."""
    for name, genes in markers.items():
        sc.tl.score_genes(adata, [gene for gene in genes if gene in adata.var_names], score_name=name)
    scores = adata.obs.groupby(key, observed=True)[list(markers)].mean()
    names = scores.idxmax(axis=1)
    return adata.obs[key].map(names).astype(str), scores


rna.obs["cell_type"], scores = name_clusters(rna)
scores.round(2).style.background_gradient(axis=1)

**Exercise 2.** Automatic names can be wrong. For each cluster, compare its name with the genes that best distinguish
it from the others: `sc.tl.rank_genes_groups(rna, "leiden")` finds them, and
`pd.DataFrame(rna.uns["rank_genes_groups"]["names"]).head(5)` lists the top five per cluster. Do you agree with every
name? If not, change it, for example `rna.obs.loc[rna.obs["leiden"] == "3", "cell_type"] = "Tumour"`.

In [ ]:
sc.tl.rank_genes_groups(rna, "leiden", method="wilcoxon")
top_genes = pd.DataFrame(rna.uns["rank_genes_groups"]["names"]).head(5).T
top_genes.insert(0, "name", rna.obs.groupby("leiden", observed=True)["cell_type"].first())
top_genes.insert(1, "cells", rna.obs["leiden"].value_counts())
top_genes

**Questions**

- Which cell types are easy to name, and which less so?
- One cluster's top genes might not match any list well. What would you do with it?

## 4. Cell types in the tissue

The table's rows link to 10x's outlines through `cell_id` (session 6), so we can draw each cell in its type's colour.
Cells left out in section 1, with too few transcripts, are in light grey.

In [ ]:
COLOURS = dict(zip(MARKERS, plt.cm.tab10.colors))
COLOURS["Too few transcripts"] = (0.85, 0.85, 0.85)


def cell_type_map(outlines, cell_types, ax, title):
    """Fill each outline (in µm) with its cell type's colour."""
    types = cell_types.astype(object).reindex(outlines.index.astype(str)).fillna("Too few transcripts").values
    for name, colour in COLOURS.items():
        chosen = outlines[types == name]
        if len(chosen):
            chosen.plot(ax=ax, color=colour)
            ax.plot([], [], "s", color=colour, label=f"{name} ({len(chosen)})")
    ax.set(xlabel="x (µm)", ylabel="y (µm)", title=title)
    ax.invert_yaxis()  # images have y pointing down
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize="small")


outlines_10x = sdata["cell_boundaries"]  # 10x's outlines are stored in µm
fig, ax = plt.subplots(figsize=(11, 9))
cell_type_map(outlines_10x, rna.obs["cell_type"], ax, "Cell types from RNA, 10x cells")
plt.show()

**Questions**

- Compare the map with the pathology annotations (session 6, section 2): do the tumour and immune regions show?
- Where are the cells with too few transcripts? Are they in one kind of tissue?

## 5. RNA and protein in the same cells

The same cells were measured twice, for RNA and for protein. If the cell types are right, the matching proteins should
be bright in them: CD3E in T cells, CD20 in B cells, CD68 and CD163 in macrophages, CD31 in endothelium, αSMA in smooth
muscle. The plot shows the mean intensity of each protein in each RNA cell type, scaled from 0 (lowest) to 1 (highest)
per protein.

In [ ]:
protein.obs["cell_type"] = rna.obs["cell_type"]
sc.pl.matrixplot(protein, protein.var_names, groupby="cell_type", layer="intensity", standard_scale="var",
                 cmap="viridis", title="Mean protein intensity per RNA cell type")

For a single marker, compare the RNA and the protein cell by cell. CD8A is measured both ways:

In [ ]:
cd8a_rna = rna[:, "CD8A"].layers["counts"].toarray().ravel()
cd8a_protein = protein[:, "CD8A"].layers["intensity"].ravel()

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(cd8a_rna + np.random.default_rng(0).uniform(-0.3, 0.3, len(cd8a_rna)), cd8a_protein, s=2, alpha=0.3)
ax.set(xlabel="CD8A transcripts per cell (jittered)", ylabel="CD8A protein intensity",
       title=f"CD8A, RNA vs protein (Spearman r = {pd.Series(cd8a_rna).corr(pd.Series(cd8a_protein), method='spearman'):.2f})")
ax.set_ylim(0, np.percentile(cd8a_protein, 99.9))  # leave out a few very bright outliers
plt.show()

Now cluster the cells on **protein alone**, as you might for an antibody-only experiment, and compare those clusters
with the RNA cell types. With 27 markers we skip the PCA. The table below shows, for each protein cluster, the
percentage of its cells in each RNA cell type; the **adjusted Rand index** summarises the agreement in one number (1:
identical groupings, 0: no better than chance).

In [ ]:
cluster(protein, use_pca=False)
agreement = adjusted_rand_score(rna.obs["cell_type"], protein.obs["leiden"])

crosstab = pd.crosstab(protein.obs["leiden"], rna.obs["cell_type"], normalize="index").mul(100)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(crosstab, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "% of the protein cluster"}, ax=ax)
ax.set(xlabel="RNA cell type", ylabel="protein cluster", title=f"Protein clusters vs RNA cell types (ARI {agreement:.2f})")
plt.show()

**Questions**

- Which RNA cell types have a clear protein cluster of their own? Which are spread over several?
- The protein panel was designed for immune cells. Which RNA cell types have no specific protein marker in it?
- Where RNA and protein disagree for one cell, which would you believe? (Think about what each one measures, and how a
  bright neighbour could affect a cell's protein intensity.)

## 6. Comparison with the Day 1 phenotypes

On Day 1 you classified cells in QuPath from their protein intensities, with single-marker thresholds combined into
phenotypes. The workshop data includes the same kind of classification for 10x's cells, made in QuPath and exported as
GeoJSON (`xenium_rcc_crop_10x_phenotypes.geojson`). Each object's `name` is its 10x cell ID, so it joins directly to
the table.

In [ ]:
def read_qupath_geojson(path):
    """QuPath GeoJSON as a GeoDataFrame in µm, with the classification name in a column."""
    objects = gpd.read_file(path).set_crs(None, allow_override=True)  # pixels, not longitude/latitude
    objects["classification"] = [c["name"] if isinstance(c, dict) else None for c in objects["classification"]]
    objects["geometry"] = objects.geometry.scale(PIXEL_SIZE, PIXEL_SIZE, origin=(0, 0))  # pixels to µm
    return objects


day1_reference = read_qupath_geojson(DATA / "qupath" / "xenium_rcc_crop_10x_phenotypes.geojson")
rna.obs["day1"] = day1_reference.set_index("name")["classification"].reindex(rna.obs_names).values

comparison = pd.crosstab(rna.obs["cell_type"], rna.obs["day1"], normalize="index").mul(100)
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(comparison, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "% of the RNA cell type"}, ax=ax)
ax.set(xlabel="Day 1 phenotype (QuPath, protein)", ylabel="cell type (RNA)", title="RNA cell types vs Day 1 phenotypes")
plt.show()

**Questions**

- Which RNA cell types match a Day 1 phenotype well? Which end up mostly as "Other" on Day 1, and why?
- Plasma cells are mostly "Other T cell" on Day 1. Look at their CD3E protein in section 5. Why might a threshold on
  CD3E call them T cells? (Think about antibodies, and about bright neighbours.)
- The Day 1 classifier can only find what its markers and thresholds describe. What did clustering find that it
  couldn't, and the other way round?

## 7. Exercise: your own cells and your own phenotypes

Now repeat the analysis on your own work:

- **your cells:** the InstanSeg cells you saved at the end of session 6 (`data/session6_cells.zarr`). Sopa's table
  holds their transcript counts in `X`, like 10x's.
- **your phenotypes:** the cells you classified in QuPath on Day 1, exported as GeoJSON in session 4
  (`data/day1_cells.geojson`).

Your cells and your Day 1 cells come from different segmentations, so they don't share names. We match them by
position instead: each of your session 6 cells gets the Day 1 phenotype of the Day 1 cell its centre falls in.

If either file is missing, the cell below uses 10x's cells (in the session 6 region) or the reference phenotypes
instead, and says so.

In [ ]:
SESSION6_REGION = [820, 720, 1260, 1160]  # x0, y0, x1, y1 in µm, as in session 6

if (DATA / "session6_cells.zarr").exists():
    own = sd.read_zarr(DATA / "session6_cells.zarr")
    own_rna = own["table"].copy()
    own_outlines = sd.transform(own["instanseg_cells"], to_coordinate_system="microns")
    print(f"Your session 6 cells: {own_rna.n_obs}")
else:
    own_outlines = outlines_10x.cx[SESSION6_REGION[0]:SESSION6_REGION[2], SESSION6_REGION[1]:SESSION6_REGION[3]]
    own_rna = sdata["table"][sdata["table"].obs["cell_id"].astype(str).isin(own_outlines.index.astype(str))].copy()
    print(f"No session 6 cells found: using 10x's {own_rna.n_obs} cells in the session 6 region instead")

if (DATA / "day1_cells.geojson").exists():
    own_day1 = read_qupath_geojson(DATA / "day1_cells.geojson")
    print(f"Your Day 1 cells: {len(own_day1)}")
else:
    own_day1 = day1_reference
    print("No Day 1 export found: using the reference phenotypes instead")


def phenotype_by_position(outlines, day1):
    """For each outline (in µm), the Day 1 phenotype of the Day 1 cell containing its centre."""
    centres = gpd.GeoDataFrame(geometry=outlines.centroid.values, index=outlines.index.astype(str))
    joined = gpd.sjoin(centres, day1[["geometry", "classification"]], predicate="within", how="left")
    return joined[~joined.index.duplicated()]["classification"]


own_phenotypes = phenotype_by_position(own_outlines, own_day1)
print(f"{own_phenotypes.notna().sum()} of {len(own_phenotypes)} cells matched to a Day 1 cell")

**Exercise 3.** With the functions from sections 1 to 6, on `own_rna`:

1. filter out cells with fewer than `MIN_TRANSCRIPTS` transcripts, normalise (`normalise_rna`), cluster (`cluster`) and
   name the clusters (`name_clusters`) into `own_rna.obs["cell_type"]`;
2. map your cell types with `cell_type_map(own_outlines, own_rna.obs["cell_type"], ax, ...)`;
3. add your Day 1 phenotypes (`own_phenotypes`) to `own_rna.obs`, and compare the two with `pd.crosstab`, as in
   section 6.

How do your results differ from 10x's cells? Is it the segmentation, the smaller number of cells, or your Day 1
choices?

In [ ]:
sc.pp.filter_cells(own_rna, min_counts=MIN_TRANSCRIPTS)
normalise_rna(own_rna)
cluster(own_rna)
own_rna.obs["cell_type"], own_scores = name_clusters(own_rna)

fig, ax = plt.subplots(figsize=(9, 8))
cell_type_map(own_outlines, own_rna.obs["cell_type"], ax, "Cell types from RNA, your cells")
plt.show()

own_rna.obs["day1"] = own_phenotypes.reindex(own_rna.obs_names).values
pd.crosstab(own_rna.obs["cell_type"], own_rna.obs["day1"].fillna("not matched"))

**Questions**

- With about 1,300 cells instead of 14,000, do you still find every cell type? Which go missing first?
- How many of your cells matched a Day 1 cell? What happens to a cell whose centre lies between two Day 1 cells?

## 8. Saving the cell types for session 8

Session 8 asks where each cell type is relative to the others. The cell below saves the cell type of every 10x cell
(and its cluster) to a CSV file.

In [ ]:
CELL_TYPES = DATA / "session7_cell_types.csv"
rna.obs[["leiden", "cell_type"]].rename_axis("cell_id").to_csv(CELL_TYPES)
print(f"Saved {rna.n_obs:,} cell types to {CELL_TYPES}")

## Summary

- Counts and intensities are different kinds of number: RNA is scaled per cell and log-transformed; protein is
  arcsinh-transformed and scaled per marker. Keep them in separate tables.
- Clustering (PCA, neighbours, Leiden) groups similar cells; the resolution sets how many groups you get, and there
  is no single right answer.
- Clusters get names from marker genes, which depend on what the panel contains; check automatic names against the
  top genes.
- RNA and protein measured in the same cells mostly agree, but each can see what the other can't, and both inherit
  the segmentation's errors.
- Threshold-based phenotypes (Day 1) and clusters (today) answer different questions: one finds what you asked for,
  the other what is there.

In session 8 we look at where the cell types are relative to each other: neighbourhoods, co-location and niches.

**Further reading**

- [scanpy tutorials](https://scanpy.readthedocs.io/en/stable/tutorials/index.html), including clustering and
  marker genes.
- Heumos et al. (2023), [Best practices for single-cell analysis across modalities](https://doi.org/10.1038/s41576-023-00586-w),
  *Nature Reviews Genetics*.